# AutoGraft 60-second demo

Entity deduplication for GraphRAG in 60 seconds.

- No Neo4j
- No API key
- No LLM call (Layer 3 never fires: the corpus stays in the non-ambiguous zone)

AutoGraft resolves 5 extracted entities against an existing graph and shows the 4-layer short-circuit: deterministic -> lexical -> semantic -> LLM arbiter (never needed here).

In [ ]:
!pip install -q autograft
!pip install -q numpy rapidfuzz pydantic

In [ ]:
import zlib

import numpy as np

from autograft.core.resolver import resolve_entity
from autograft.layers.semantic import cosine_similarity
from autograft.models.entities import Entity, ExistingNode

DIM = 384  # all-MiniLM-L6-v2 embedding size, fully deterministic, no model needed


def _unit(seed: str) -> np.ndarray:
    rng = np.random.default_rng(zlib.crc32(seed.encode()))
    vec = rng.standard_normal(DIM)
    return vec / float(np.linalg.norm(vec))


def _blend(parts: list[tuple[np.ndarray, float]]) -> list[float]:
    vec = sum(w * v for v, w in parts)
    if not parts:
        return [0.0] * DIM
    return [float(x) for x in vec / float(np.linalg.norm(vec))]


_APPLE = _unit("concept:apple")
_COMPANY = _unit("concept:company")
_LANGUAGE = _unit("concept:language")
_SNAKE = _unit("concept:snake")
_ANIMAL = _unit("concept:animal")

APPLE_INC_EMB = _blend([(_APPLE, 1.0), (_COMPANY, 0.6)])
APPLE_EMB = _blend([(_APPLE, 1.0), (_COMPANY, 0.3)])
PY_LANG_EMB = _blend([(_LANGUAGE, 1.0), (_SNAKE, 0.5)])
PY_ANIMAL_EMB = _blend([(_SNAKE, 1.0), (_ANIMAL, 0.7)])

In [ ]:
graph = [
    ExistingNode(node_id="n-apple", canonical_name="Apple Inc.", type="Organization", embedding=APPLE_INC_EMB),
    ExistingNode(node_id="n-python-lang", canonical_name="Python", type="ProgrammingLanguage", embedding=PY_LANG_EMB),
]

extracted = [
    Entity(canonical_name="Apple Inc.", type="Organization", embedding=APPLE_INC_EMB),
    Entity(canonical_name="Apple", type="Organization", embedding=APPLE_EMB),
    Entity(canonical_name="Apple Incorporated", type="Organization", embedding=APPLE_INC_EMB),
    Entity(canonical_name="Python", type="ProgrammingLanguage", embedding=PY_LANG_EMB),
    Entity(canonical_name="Python", type="Animal", embedding=PY_ANIMAL_EMB),
]

total_tokens = 0
print(f"Existing graph: {len(graph)} clean nodes\n")
for i, entity in enumerate(extracted, start=1):
    result = resolve_entity(entity, db_client=graph)
    total_tokens += result.tokens_used
    matched = next((n for n in graph if n.node_id == result.matched_node_id), None)
    semantic = (
        cosine_similarity(entity.embedding or [], matched.embedding or [])
        if matched and entity.embedding and matched.embedding
        else 0.0
    )
    if result.is_match:
        layer = f"{result.layer:11s} (score {result.score:5.1f})"
        outcome = f"merged into {result.matched_node_id}"
    else:
        layer = "declined"
        outcome = "new node (type gate: no candidate)"
    sem = f"cos {semantic:.2f}" if semantic else "cos --"
    print(f"{i}. {entity.canonical_name:20s} ({entity.type:20s}) -> {layer}  {sem}  tokens {result.tokens_used}  {outcome}")

print(f"\nTotal LLM tokens used: {total_tokens}  (Layer 3 never fired)")
print("\nGraph BEFORE (naive insert): 7 nodes -> Graph AFTER (AutoGraft): 3 nodes")

## What just happened

| # | Entity | Layer that resolved it |
|---|--------|-----------------------|
| 1 | `Apple Inc.` | Layer 1 deterministic (score 100.0) |
| 2 | `Apple` | Layer 1.5 lexical, suffix-strip `Apple Inc.` -> `apple` (score 100.0) |
| 3 | `Apple Incorporated` | Layer 2 semantic, embedding cosine 1.0 (the suffix-strip misses `Incorporated`, so Layer 2 catches it) |
| 4 | `Python` | Layer 1 deterministic (score 100.0) |
| 5 | `Python` (Animal) | Declined: the type gate keeps it out of `ProgrammingLanguage`, and its embedding (cos 0.43 < 0.75) would make Layer 2 decline too |

**Before:** a naive insert would leave 7 nodes. **After:** 3 clean nodes.

**Cost:** 0 LLM calls, 0 tokens across the whole run. Layer 3 (LLM arbiter) only activates on genuinely ambiguous `semantic_uncertain` results, so it never fired.

See the full source at https://github.com/jules-gd-dev/autograft-lib/blob/master/examples/demo_60s.py